In [ ]:
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, KeepTogether
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4, landscape

styles = getSampleStyleSheet()

# 제목 스타일 커스터마이즈
title_style = ParagraphStyle(
    "CustomTitle",
    parent=styles["Heading2"],
    fontSize=14,
    leading=18,
    textColor=colors.HexColor("#003366"),  # 짙은 파랑
    spaceAfter=10,
)

doc = SimpleDocTemplate(
    "stats_summary.pdf",
    pagesize=landscape(A4),
    leftMargin=18, rightMargin=18, topMargin=18, bottomMargin=18
)

elements = []

def df_to_table_fit(df, title):
    df_show = df.reset_index()

    # datetime 컬럼 깔끔하게 변환
    for col in df_show.select_dtypes(include=["datetime64[ns]"]).columns:
        df_show[col] = df_show[col].dt.strftime("%Y-%m-%d")

    data = [df_show.columns.tolist()] + df_show.values.tolist()

    # 열폭 균등 배분
    avail_width = doc.pagesize[0] - doc.leftMargin - doc.rightMargin
    ncols = len(df_show.columns)
    col_widths = [avail_width / ncols] * ncols

    table = Table(data, colWidths=col_widths, repeatRows=1)

    # 테이블 스타일
    style = TableStyle([
        ("FONTSIZE", (0,0), (-1,-1), 7),
        ("LEADING", (0,0), (-1,-1), 8),

        # 헤더 스타일
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#003366")),
        ("TEXTCOLOR", (0,0), (-1,0), colors.white),
        ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),

        # 줄무늬 (zebra stripe)
        ("BACKGROUND", (0,1), (-1,-1), colors.whitesmoke),
        ("BACKGROUND", (0,2), (-1,-1,2), colors.lightgrey),

        # 테두리
        ("GRID", (0,0), (-1,-1), 0.3, colors.black),

        # 정렬
        ("ALIGN", (1,1), (-1,-1), "RIGHT"),   # 숫자는 오른쪽
        ("ALIGN", (0,0), (0,-1), "LEFT"),     # 첫 열(예: 변수명)은 왼쪽
    ])
    table.setStyle(style)

    # 제목 + 표 묶기
    title_para = Paragraph(title, title_style)
    return KeepTogether([title_para, table, Spacer(1, 12)])

# PDF에 넣기
elements.append(df_to_table_fit(stats.round(3), "📊 Summary Statistics"))
elements.append(df_to_table_fit(nstocks_per_p.round(3), "📈 Average Number of Stocks per Portfolio"))
elements.append(df_to_table_fit(assignment2_data.head(25), "📂 Assignment2 Data (First 25 rows)"))

doc.build(elements)
print("PDF 저장 완료: stats_summary.pdf")
